# 10 · Validasi Label dengan Penilaian Manusia (Opsional)
Lihat `laporan/arsip/2026-11-07_sebelum_run/catatan/kritik-reviewer.md` Bagian 3.2. Tujuannya menjawab dua pertanyaan:
1. Target mana yang lebih dekat dengan penilaian manusia, T1 (suara mentah) atau T3 (terkoreksi usia)?
2. Apakah urutan dari model (U, B5, B0) cocok dengan urutan menurut manusia?

**Alur:** (1) buat lembar anotasi dari data uji → (2) dua sampai tiga anggota menilai secara terpisah tanpa melihat jumlah suara,
tanggal, atau skor model → (3) simpan hasilnya sebagai `data/human/annotator_<nama>.csv` → (4) jalankan bagian analisis.

> **Google Colab.** Jalankan sekali per sesi sebelum sel setup:
> ```
> !git clone -b main https://github.com/Vito28/dl-paper.git
> %cd dl-paper/notebooks
> !pip install -q -r ../requirements.txt
> ```
> Sel setup otomatis me-mount Google Drive (izinkan akses saat diminta). **Semua data dan hasil, termasuk
> unduhan mentah, disimpan di `MyDrive/dl-paper`**, bukan di instance. Jika Drive gagal di-mount atau
> terputus, notebook berhenti dengan error. Jalankan sel terakhir (`finish()`) agar semua tulisan tersinkron.

In [ ]:
# Setup: path proyek dan konfigurasi bersama (ubah konfigurasi di notebooks/settings.py)
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "notebooks")]
from settings import cfg, SMOKE, finish
import numpy as np, pandas as pd
pd.set_option("display.width", 160)
print("kategori:", cfg.category, "| output:", cfg.out_dir, "| SMOKE:", SMOKE)

In [ ]:
from hr import experiment as X, human, metrics
b = X.load_bundle(cfg)
te = b.rows("test")
HUMAN = cfg.data_dir / "human"; HUMAN.mkdir(parents=True, exist_ok=True)
print(human.RUBRIC)

## 1. Buat lembar anotasi

In [ ]:
sheet_path, key_path = HUMAN / "lembar_anotasi.csv", HUMAN / "KUNCI_JANGAN_DIBUKA.csv"
if not sheet_path.exists():
    blind, key = human.make_sheet(te, n_products=80 if not SMOKE else 12, per_product=6)
    blind.to_csv(sheet_path, index=False); key.to_csv(key_path, index=False)
blind = pd.read_csv(sheet_path); key = pd.read_csv(key_path)
print(len(blind), "ulasan dari", blind["product_code"].nunique(), "produk"); blind.head()

## 2. Analisis setelah anotasi selesai
Salin `lembar_anotasi.csv` untuk setiap penilai, isi kolom `score_1_5`, lalu simpan sebagai `annotator_<nama>.csv` di folder yang sama.

In [ ]:
files = sorted(HUMAN.glob("annotator_*.csv"))
if SMOKE and not files:   # simulasi dua penilai untuk uji cepat
    rng = np.random.default_rng(0)
    base = rng.integers(1, 6, len(blind))
    for n in ("a", "b"):
        f = blind.copy(); f["score_1_5"] = np.clip(base + rng.integers(-1, 2, len(f)), 1, 5)
        f.to_csv(HUMAN / f"annotator_{n}.csv", index=False)
    files = sorted(HUMAN.glob("annotator_*.csv"))
print([f.name for f in files])

In [ ]:
if len(files) >= 2:
    A = pd.concat([pd.read_csv(f)[["item_id", "score_1_5"]].rename(columns={"score_1_5": f.stem}) for f in files], axis=1)
    A = A.loc[:, ~A.columns.duplicated()].set_index("item_id")
    R_ = A.to_numpy(dtype=float).T
    agree = {"krippendorff_alpha_interval": human.krippendorff_alpha_interval(R_)}
    if len(files) == 2:
        agree["cohen_kappa_kuadratik"] = human.weighted_kappa(R_[0], R_[1])
    cfg.path("tables", "manusia_kesepakatan.json").write_text(json.dumps(agree, indent=2)); print(agree)
else:
    print(f"Analisis DILEWATI: butuh >= 2 berkas annotator_*.csv di {HUMAN}, ditemukan {len(files)}.")

### Target dan model mana yang paling cocok dengan penilaian manusia?
Rata-rata nilai penilai menjadi "kebenaran" untuk setiap ulasan. Lalu dihitung korelasi Spearman per produk antara nilai manusia dan T1, T3, serta skor model.

In [ ]:
if len(files) >= 2:
    hm = A.mean(axis=1).rename("human").reset_index().merge(key, on="item_id")
    d = hm.merge(b.df[["rid", "parent_asin", "y_t1", "y_t3", "review_length"]], on="rid")
    cands = {"T1 (suara mentah)": d["y_t1"], "T3 (terkoreksi usia)": d["y_t3"], "B0 panjang ulasan": d["review_length"]}
    for c in ["U", "B5"]:
        runs = [r for r in X.results_table(cfg)["run_id"] if r.startswith(f"{c}_{cfg.main_scale}_T1_s")]
        if runs:
            cands[f"model {c}"] = d[["rid"]].merge(X.load_pred(cfg, runs[0], "test"), on="rid", how="left")["score"]
    hg = np.clip(np.round(d["human"]) - 1, 0, 4).astype(int).to_numpy()   # tingkat 0..4 dari nilai 1..5
    rows = []
    for name, s in cands.items():
        pp = metrics.per_product_metrics(d["parent_asin"].to_numpy(), s.to_numpy(), hg, d["human"].to_numpy(),
                                         d["rid"].to_numpy(), map_rel=3)
        rows.append({"pembanding": name, **metrics.macro(pp)})
    hum_tab = pd.DataFrame(rows); hum_tab.to_csv(cfg.path("tables", "manusia_vs_target_model.csv"), index=False)
    display(hum_tab.round(4))
else:
    print(f"Analisis DILEWATI: butuh >= 2 berkas annotator_*.csv di {HUMAN}, ditemukan {len(files)}.")

## Simpan ke Google Drive
Memastikan semua hasil notebook ini sudah tersinkron ke Drive sebelum runtime ditutup.

In [ ]:
finish()